# EN4554 – Assignment 2: Convolutional Neural Networks

**Name:** John Praveen  &nbsp;&nbsp; **Index No:** `220491B` *(fill in)*  
**GitHub:** https://github.com/john-praveen/EN4554-Assignment-2

**Contents:** Setup · Task 1 Backpropagation · Task 2 Optimizers · Task 3 Regularization & residual connections · Task 4 Best CNN (optional)

## 0. Setup

This section covers the environment, the fixed seeds, and the CIFAR-10 data pipeline. CIFAR-10 is split 45k train / 5k validation, and each channel is normalised with the **training-split** mean and std. The test set is not touched until Task 4.

In [ ]:
import random, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import torchvision, matplotlib.pyplot as plt

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

set_seed()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

In [ ]:
# CIFAR-10: 45k train / 5k validation, normalised with training statistics
train_data = torchvision.datasets.CIFAR10('./data', train=True, download=True)
X = torch.tensor(train_data.data).permute(0, 3, 1, 2).float() / 255   # (50000, 3, 32, 32)
y = torch.tensor(train_data.targets)

perm = torch.randperm(50000)
X_tr, y_tr = X[perm[:45000]], y[perm[:45000]]
X_va, y_va = X[perm[45000:]], y[perm[45000:]]

mean = X_tr.mean(dim=(0, 2, 3), keepdim=True)   # per-channel mean/std, training split only
std  = X_tr.std(dim=(0, 2, 3), keepdim=True)
X_tr = ((X_tr - mean) / std).to(device); y_tr = y_tr.to(device)
X_va = ((X_va - mean) / std).to(device); y_va = y_va.to(device)
print(X_tr.shape, X_va.shape)

In [ ]:
def accuracy(model, X, y):
    model.eval()
    with torch.no_grad():
        return (model(X).argmax(1) == y).float().mean().item()

**Notes.** The data is kept on the GPU as plain tensors, so no `DataLoader` is needed. The mean and std come from the 45k training images only, so nothing leaks from validation or test. The test set is left untouched until Task 4.